# Tackling Mode Collapse in GANs: DCGAN vs WGAN-GP

**Objective:** Design and implement a GAN system that addresses mode collapse by comparing a
**Baseline Deep Convolutional GAN (DCGAN)** against a **Wasserstein GAN with Gradient Penalty (WGAN-GP)**.

We demonstrate how the Wasserstein loss with gradient penalty dramatically improves training stability
and the diversity of generated images compared to the standard BCE-based DCGAN.

| Property | DCGAN | WGAN-GP |
|---|---|---|
| Loss | Binary Cross-Entropy | Wasserstein Distance |
| Discriminator output | Sigmoid (probability) | Raw score (critic) |
| Regularisation | — | Gradient Penalty (λ=10) |
| Critic updates / Gen update | 1 | 5 |
| Mode collapse risk | High | Low |

**Dataset:** Anime Faces 64×64  
**Platform:** Kaggle GPU T4 ×2

In [ ]:
# ============================================================
# 1. Environment Setup & Installs
# ============================================================
# Kaggle Internet must be enabled for Gradio install
!pip install -q gradio

import os, sys, random, time, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  "
              f"Memory: {torch.cuda.get_device_properties(i).total_mem / 1024**3:.1f} GB")

In [ ]:
# ============================================================
# 2. Configuration & Hyperparameters
# ============================================================
# ── Dataset paths (update for your Kaggle input) ──
# Anime Faces : /kaggle/input/anime-faces/
# Pokemon      : /kaggle/input/pokemon-sprites/
DATA_PATH = "/kaggle/input/anime-faces/"

# ── Architecture ──
IMAGE_SIZE  = 64        # Spatial size of training images
NC          = 3         # Number of channels (RGB)
Z_DIM       = 100       # Size of z latent vector
NGF         = 64        # Generator feature map depth
NDF         = 64        # Discriminator/Critic feature map depth

# ── Training ──
BATCH_SIZE  = 64
NUM_EPOCHS  = 30        # Increase for better results; 30 is a good Kaggle baseline
LR          = 0.0002    # Adam learning rate
BETA1       = 0.5       # Adam beta1
BETA2       = 0.999     # Adam beta2

# ── WGAN-GP specific ──
N_CRITIC    = 5         # Critic updates per generator update
LAMBDA_GP   = 10        # Gradient penalty coefficient

# ── Mixed Precision ──
USE_AMP     = True

# ── Checkpointing ──
SAVE_EVERY  = 5         # Save checkpoints every N epochs
OUTPUT_DIR  = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Device ──
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"\nPrimary device: {device}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

# 3. Data Preparation
- Resize all images to 64 × 64
- Normalize to [-1, 1] (required for Tanh generator output)
- Use a custom dataset class that recursively finds images so it works regardless of dataset folder structure

In [ ]:
# ============================================================
# 3. Data Preparation
# ============================================================
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),       # light augmentation
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),          # -> [-1, 1]
                         (0.5, 0.5, 0.5)),
])

class ImageFolderFlat(data.Dataset):
    """Recursively load every image under *root*, ignoring subfolder structure."""
    EXTENSIONS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}

    def __init__(self, root, transform=None):
        self.transform = transform
        self.paths = []
        for dirpath, _, filenames in os.walk(root):
            for fname in sorted(filenames):
                if os.path.splitext(fname)[1].lower() in self.EXTENSIONS:
                    self.paths.append(os.path.join(dirpath, fname))
        print(f"Found {len(self.paths)} images under {root}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, 0   # dummy label

dataset = ImageFolderFlat(DATA_PATH, transform=transform)
assert len(dataset) > 0, f"No images found in {DATA_PATH} — check the path!"

dataloader = data.DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,         # keep batch sizes uniform
)
print(f"Total batches per epoch: {len(dataloader)}")

In [ ]:
# ── Visualise a batch of real training images ──
real_batch = next(iter(dataloader))[0]
grid = vutils.make_grid(real_batch[:64], nrow=8, padding=2, normalize=True)
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Sample Training Images (Anime Faces)", fontsize=14)
plt.imshow(np.transpose(grid.cpu().numpy(), (1, 2, 0)))
plt.tight_layout()
plt.show()

# 4. DCGAN Architecture

## Generator
| Layer | Type | Output Shape |
|-------|------|-------------|
| 1 | ConvTranspose2d + BN + ReLU | 512 × 4 × 4 |
| 2 | ConvTranspose2d + BN + ReLU | 256 × 8 × 8 |
| 3 | ConvTranspose2d + BN + ReLU | 128 × 16 × 16 |
| 4 | ConvTranspose2d + BN + ReLU | 64 × 32 × 32 |
| 5 | ConvTranspose2d + Tanh       | 3 × 64 × 64 |

## Discriminator
| Layer | Type | Output Shape |
|-------|------|-------------|
| 1 | Conv2d + LeakyReLU            | 64 × 32 × 32 |
| 2 | Conv2d + BN + LeakyReLU       | 128 × 16 × 16 |
| 3 | Conv2d + BN + LeakyReLU       | 256 × 8 × 8 |
| 4 | Conv2d + BN + LeakyReLU       | 512 × 4 × 4 |
| 5 | Conv2d + Sigmoid              | 1 × 1 × 1 |

In [ ]:
# ============================================================
# 4. Model Definitions
# ============================================================

def weights_init(m):
    """Custom weight initialisation (DCGAN paper)."""
    cn = m.__class__.__name__
    if 'Conv' in cn:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cn or 'InstanceNorm' in cn:
        if m.weight is not None:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)


# ────────────────── Generator (shared by DCGAN & WGAN-GP) ──────────────────
class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM, ngf=NGF, nc=NC):
        super().__init__()
        self.main = nn.Sequential(
            # z -> (ngf*8) x 4 x 4
            nn.ConvTranspose2d(z_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # -> (ngf*4) x 8 x 8
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # -> (ngf*2) x 16 x 16
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # -> (ngf) x 32 x 32
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # -> (nc) x 64 x 64
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, x):
        return self.main(x)


# ────────────────── DCGAN Discriminator (Sigmoid output) ──────────────────
class Discriminator(nn.Module):
    def __init__(self, nc=NC, ndf=NDF):
        super().__init__()
        self.main = nn.Sequential(
            # -> (ndf) x 32 x 32
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (ndf*2) x 16 x 16
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (ndf*4) x 8 x 8
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (ndf*8) x 4 x 4
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # -> 1 x 1 x 1
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            # nn.Sigmoid(),  # Replaced internally by BCEWithLogitsLoss for AMP compatibility
        )

    def forward(self, x):
        return self.main(x).view(-1)


# ────────────────── WGAN-GP Critic (NO Sigmoid) ──────────────────
class Critic(nn.Module):
    """Wasserstein critic – identical to Discriminator but:
    • No Sigmoid output (raw scores)
    • InstanceNorm instead of BatchNorm (gradient penalty requires
      per-sample independence)
    """
    def __init__(self, nc=NC, ndf=NDF):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf * 2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf * 8, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            # NO Sigmoid
        )

    def forward(self, x):
        return self.main(x).view(-1)

In [ ]:
# ── Instantiate all four networks ──
netG_dc = Generator().to(device)
netD_dc = Discriminator().to(device)
netG_wg = Generator().to(device)
netC_wg = Critic().to(device)

# Multi-GPU support (Kaggle T4 x2)
if torch.cuda.device_count() > 1:
    print(f"Wrapping models with DataParallel ({torch.cuda.device_count()} GPUs)")
    netG_dc = nn.DataParallel(netG_dc)
    netD_dc = nn.DataParallel(netD_dc)
    netG_wg = nn.DataParallel(netG_wg)
    netC_wg = nn.DataParallel(netC_wg)

# Custom weight init
for net in [netG_dc, netD_dc, netG_wg, netC_wg]:
    net.apply(weights_init)

# Print model summaries
print("=" * 60)
print("GENERATOR (shared architecture)")
print("=" * 60)
print(netG_dc)
print(f"\nParameters: {sum(p.numel() for p in netG_dc.parameters()):,}")
print("\n" + "=" * 60)
print("DCGAN DISCRIMINATOR")
print("=" * 60)
print(netD_dc)
print(f"\nParameters: {sum(p.numel() for p in netD_dc.parameters()):,}")
print("\n" + "=" * 60)
print("WGAN-GP CRITIC")
print("=" * 60)
print(netC_wg)
print(f"\nParameters: {sum(p.numel() for p in netC_wg.parameters()):,}")

# 5. Training Utilities
Helper functions for:
- GPU memory monitoring
- Gradient penalty computation (WGAN-GP)
- Image grid saving

In [ ]:
# ============================================================
# 5. Training Utilities
# ============================================================

def gpu_mem_report():
    """Print current GPU memory usage."""
    if not torch.cuda.is_available():
        return
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1024**2
        resrv = torch.cuda.memory_reserved(i) / 1024**2
        print(f"  GPU {i}: {alloc:.0f} MB allocated / {resrv:.0f} MB reserved")


def compute_gradient_penalty(critic, real, fake, device):
    """
    Gradient penalty for WGAN-GP.
    Must run in float32 (not inside autocast) because autograd.grad
    with create_graph=True is incompatible with AMP.
    """
    bs = real.size(0)
    alpha = torch.rand(bs, 1, 1, 1, device=device)
    interpolated = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    
    # Forward pass on interpolated samples (full precision)
    scores = critic(interpolated)
    
    grad_outputs = torch.ones_like(scores, device=device)
    gradients = torch.autograd.grad(
        outputs=scores,
        inputs=interpolated,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    
    gradients = gradients.view(bs, -1)
    penalty = LAMBDA_GP * ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return penalty


def save_image_grid(tensor, path, nrow=8):
    """Save a batch of images as a grid."""
    grid = vutils.make_grid(tensor, nrow=nrow, padding=2, normalize=True)
    ndarr = grid.mul(255).add_(0.5).clamp_(0, 255).permute(1, 2, 0).cpu().numpy().astype(np.uint8)
    Image.fromarray(ndarr).save(path)


# Fixed noise vectors (same for both models so we can compare evolution)
fixed_noise = torch.randn(64, Z_DIM, 1, 1, device=device)
print("Utilities ready. Initial GPU state:")
gpu_mem_report()

# 6. Training — Baseline DCGAN
- **Loss:** Binary Cross-Entropy
- **Label smoothing:** real label = 0.9 (reduces overconfidence and improves stability)
- **Mixed Precision:** `torch.cuda.amp`
- **Checkpoints** saved every 5 epochs

In [ ]:
# ============================================================
# 6. Train DCGAN
# ============================================================
criterion = nn.BCEWithLogitsLoss()
optD = optim.Adam(netD_dc.parameters(), lr=LR, betas=(BETA1, BETA2))
optG = optim.Adam(netG_dc.parameters(), lr=LR, betas=(BETA1, BETA2))

scaler_d = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler_g = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# ── Tracking ──
hist_dc = {"G": [], "D": [], "D_x": [], "D_Gz": [],
           "G_epoch": [], "D_epoch": []}
img_snapshots_dc = []

REAL_LABEL = 0.9   # one-sided label smoothing
FAKE_LABEL = 0.0

print("Starting DCGAN training...")
t0 = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_g, epoch_d, n_batches = 0.0, 0.0, 0

    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        bs = real_imgs.size(0)

        # ── (1) Update Discriminator ─────────────────────────
        netD_dc.zero_grad()

        # Real batch
        labels_real = torch.full((bs,), REAL_LABEL, device=device)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out_real = netD_dc(real_imgs)
            loss_d_real = criterion(out_real, labels_real)
        scaler_d.scale(loss_d_real).backward()
        D_x = torch.sigmoid(out_real).mean().item()

        # Fake batch
        noise = torch.randn(bs, Z_DIM, 1, 1, device=device)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            fake = netG_dc(noise)
            labels_fake = torch.full((bs,), FAKE_LABEL, device=device)
            out_fake = netD_dc(fake.detach())
            loss_d_fake = criterion(out_fake, labels_fake)
        scaler_d.scale(loss_d_fake).backward()
        D_Gz1 = torch.sigmoid(out_fake).mean().item()

        loss_d = loss_d_real + loss_d_fake
        scaler_d.step(optD)
        scaler_d.update()

        # ── (2) Update Generator ─────────────────────────────
        netG_dc.zero_grad()
        labels_real.fill_(1.0)  # generator wants D to output 1
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out_fake2 = netD_dc(fake)
            loss_g = criterion(out_fake2, labels_real)
        scaler_g.scale(loss_g).backward()
        D_Gz2 = torch.sigmoid(out_fake2).mean().item()
        scaler_g.step(optG)
        scaler_g.update()

        # ── Logging ──
        hist_dc["G"].append(loss_g.item())
        hist_dc["D"].append(loss_d.item())
        hist_dc["D_x"].append(D_x)
        hist_dc["D_Gz"].append(D_Gz2)
        epoch_g += loss_g.item()
        epoch_d += loss_d.item()
        n_batches += 1

        if i % 100 == 0:
            print(f"  [Epoch {epoch+1}/{NUM_EPOCHS}][{i}/{len(dataloader)}]  "
                  f"D: {loss_d.item():.4f}  G: {loss_g.item():.4f}  "
                  f"D(x): {D_x:.3f}  D(G(z)): {D_Gz1:.3f}/{D_Gz2:.3f}")

    hist_dc["G_epoch"].append(epoch_g / n_batches)
    hist_dc["D_epoch"].append(epoch_d / n_batches)

    # ── Snapshot generated images ──
    with torch.no_grad():
        snap = netG_dc(fixed_noise).detach().cpu()
    img_snapshots_dc.append(vutils.make_grid(snap, nrow=8, padding=2, normalize=True))

    # ── Checkpoint ──
    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save(netG_dc.state_dict(), os.path.join(OUTPUT_DIR, f"dcgan_G_ep{epoch+1}.pth"))
        torch.save(netD_dc.state_dict(), os.path.join(OUTPUT_DIR, f"dcgan_D_ep{epoch+1}.pth"))
        save_image_grid(snap, os.path.join(OUTPUT_DIR, f"dcgan_samples_ep{epoch+1}.png"))
        print(f"  ✓ Checkpoint saved (epoch {epoch+1})")
        gpu_mem_report()

elapsed = time.time() - t0
torch.save(netG_dc.state_dict(), os.path.join(OUTPUT_DIR, "dcgan_G_final.pth"))
torch.save(netD_dc.state_dict(), os.path.join(OUTPUT_DIR, "dcgan_D_final.pth"))
print(f"\n✔ DCGAN training complete — {elapsed/60:.1f} min")

# 7. Training — WGAN-GP (Advanced Model)
Key differences from DCGAN:
- **Loss:** Wasserstein distance (no log, no sigmoid)
- **Gradient Penalty** enforces 1-Lipschitz constraint (λ = 10)
- **Critic trained 5× per generator step** for a tighter Wasserstein estimate
- **No label smoothing** needed — Wasserstein loss is inherently stable
- Gradient penalty computed in **full precision** (required — AMP autocast is incompatible with `torch.autograd.grad`)

In [ ]:
# ============================================================
# 7. Train WGAN-GP
# ============================================================
optC = optim.Adam(netC_wg.parameters(), lr=LR, betas=(BETA1, BETA2))
optG_wg = optim.Adam(netG_wg.parameters(), lr=LR, betas=(BETA1, BETA2))

scaler_c = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler_gw = torch.cuda.amp.GradScaler(enabled=USE_AMP)

hist_wg = {"G": [], "C": [], "W_dist": [],
           "G_epoch": [], "C_epoch": [], "W_epoch": []}
img_snapshots_wg = []

print("Starting WGAN-GP training...")
t0 = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_g, epoch_c, epoch_w, n_batches = 0.0, 0.0, 0.0, 0

    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        bs = real_imgs.size(0)

        # ── (1) Update Critic (N_CRITIC times) ────────────────
        for _ in range(N_CRITIC):
            netC_wg.zero_grad()
            noise = torch.randn(bs, Z_DIM, 1, 1, device=device)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                fake_imgs = netG_wg(noise).detach()
                score_real = netC_wg(real_imgs)
                score_fake = netC_wg(fake_imgs)
                wasserstein_d = score_real.mean() - score_fake.mean()
                c_loss_base = -wasserstein_d   # critic wants to MAXIMISE W distance

            # Gradient penalty must be computed in float32
            gp = compute_gradient_penalty(
                netC_wg, real_imgs.float(), fake_imgs.float(), device
            )
            c_loss = c_loss_base + gp
            # Cannot use scaler for the GP part; just backward normally
            c_loss.backward()
            optC.step()

        # ── (2) Update Generator ─────────────────────────────
        netG_wg.zero_grad()
        noise_g = torch.randn(bs, Z_DIM, 1, 1, device=device)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            gen_imgs = netG_wg(noise_g)
            g_loss = -netC_wg(gen_imgs).mean()

        scaler_gw.scale(g_loss).backward()
        scaler_gw.step(optG_wg)
        scaler_gw.update()

        # ── Logging ──
        hist_wg["G"].append(g_loss.item())
        hist_wg["C"].append(c_loss.item())
        hist_wg["W_dist"].append(wasserstein_d.item())
        epoch_g += g_loss.item()
        epoch_c += c_loss.item()
        epoch_w += wasserstein_d.item()
        n_batches += 1

        if i % 100 == 0:
            print(f"  [Epoch {epoch+1}/{NUM_EPOCHS}][{i}/{len(dataloader)}]  "
                  f"C: {c_loss.item():.4f}  G: {g_loss.item():.4f}  "
                  f"W_dist: {wasserstein_d.item():.4f}")

    hist_wg["G_epoch"].append(epoch_g / n_batches)
    hist_wg["C_epoch"].append(epoch_c / n_batches)
    hist_wg["W_epoch"].append(epoch_w / n_batches)

    # ── Snapshot ──
    with torch.no_grad():
        snap = netG_wg(fixed_noise).detach().cpu()
    img_snapshots_wg.append(vutils.make_grid(snap, nrow=8, padding=2, normalize=True))

    # ── Checkpoint ──
    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save(netG_wg.state_dict(), os.path.join(OUTPUT_DIR, f"wgan_G_ep{epoch+1}.pth"))
        torch.save(netC_wg.state_dict(), os.path.join(OUTPUT_DIR, f"wgan_C_ep{epoch+1}.pth"))
        save_image_grid(snap, os.path.join(OUTPUT_DIR, f"wgan_samples_ep{epoch+1}.png"))
        print(f"  ✓ Checkpoint saved (epoch {epoch+1})")
        gpu_mem_report()

elapsed = time.time() - t0
torch.save(netG_wg.state_dict(), os.path.join(OUTPUT_DIR, "wgan_G_final.pth"))
torch.save(netC_wg.state_dict(), os.path.join(OUTPUT_DIR, "wgan_C_final.pth"))
print(f"\n✔ WGAN-GP training complete — {elapsed/60:.1f} min")

# 8. Visualisation & Evaluation

## 8.1 Loss Curves
Plotting both per-iteration (faded) and epoch-averaged (bold) losses for easy comparison.

In [ ]:
# ============================================================
# 8.1  Loss Plots
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ── DCGAN losses ──
ax = axes[0]
ax.set_title("DCGAN — Loss per Iteration", fontsize=13)
ax.plot(hist_dc["G"], alpha=0.25, color="tab:blue")
ax.plot(hist_dc["D"], alpha=0.25, color="tab:orange")
ax.set_xlabel("Iteration"); ax.set_ylabel("Loss")
ax.legend(["Generator", "Discriminator"], loc="upper right")

# ── DCGAN epoch averages ──
ax = axes[1]
ax.set_title("DCGAN — Epoch-Averaged Loss", fontsize=13)
ax.plot(hist_dc["G_epoch"], marker="o", color="tab:blue", label="Generator")
ax.plot(hist_dc["D_epoch"], marker="s", color="tab:orange", label="Discriminator")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend()

# ── WGAN-GP epoch averages ──
ax = axes[2]
ax.set_title("WGAN-GP — Epoch-Averaged Loss", fontsize=13)
ax.plot(hist_wg["G_epoch"], marker="o", color="tab:green", label="Generator")
ax.plot(hist_wg["C_epoch"], marker="s", color="tab:red", label="Critic")
ax2 = ax.twinx()
ax2.plot(hist_wg["W_epoch"], marker="^", color="tab:purple", label="W Distance")
ax2.set_ylabel("Wasserstein Distance", color="tab:purple")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curves.png"), dpi=150)
plt.show()
print("Loss curves saved to loss_curves.png")

In [ ]:
# ── D(x) and D(G(z)) over iterations (DCGAN only) ──
fig, ax = plt.subplots(figsize=(10, 4))
ax.set_title("DCGAN — D(x) and D(G(z)) over Training", fontsize=13)
ax.plot(hist_dc["D_x"], alpha=0.3, color="tab:green", label="D(x)")
ax.plot(hist_dc["D_Gz"], alpha=0.3, color="tab:red", label="D(G(z))")
# rolling average
window = min(50, len(hist_dc["D_x"]))
if window > 1:
    from numpy import convolve
    kernel = np.ones(window) / window
    ax.plot(convolve(hist_dc["D_x"], kernel, mode='valid'), color="tab:green", linewidth=2)
    ax.plot(convolve(hist_dc["D_Gz"], kernel, mode='valid'), color="tab:red", linewidth=2)
ax.axhline(0.5, ls='--', color='gray', alpha=0.5, label="Ideal D(x)≈0.5")
ax.set_xlabel("Iteration"); ax.set_ylabel("Probability")
ax.legend(); ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "dcgan_Dx_DGz.png"), dpi=150)
plt.show()

## 8.2 Generated Image Comparison

In [ ]:
# ============================================================
# 8.2  Final generated samples — side by side
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].set_title("DCGAN — Final Epoch Samples", fontsize=14)
axes[0].axis("off")
if img_snapshots_dc:
    axes[0].imshow(np.transpose(img_snapshots_dc[-1].numpy(), (1, 2, 0)))

axes[1].set_title("WGAN-GP — Final Epoch Samples", fontsize=14)
axes[1].axis("off")
if img_snapshots_wg:
    axes[1].imshow(np.transpose(img_snapshots_wg[-1].numpy(), (1, 2, 0)))

plt.suptitle("Side-by-Side: DCGAN vs WGAN-GP (same fixed noise)", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_final.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8.3 Training Progression
How generated images evolve across epochs (every 5th epoch snapshot).

In [ ]:
# ============================================================
# 8.3  Training progression grid
# ============================================================
def plot_progression(snapshots, title, save_name):
    n = len(snapshots)
    step = max(1, n // 6)  # show ~6 snapshots
    indices = list(range(0, n, step))
    if (n - 1) not in indices:
        indices.append(n - 1)

    fig, axes = plt.subplots(1, len(indices), figsize=(4 * len(indices), 4))
    if len(indices) == 1:
        axes = [axes]
    for ax, idx in zip(axes, indices):
        ax.set_title(f"Epoch {idx + 1}", fontsize=11)
        ax.axis("off")
        ax.imshow(np.transpose(snapshots[idx].numpy(), (1, 2, 0)))
    plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150, bbox_inches="tight")
    plt.show()

plot_progression(img_snapshots_dc, "DCGAN — Training Progression", "dcgan_progression.png")
plot_progression(img_snapshots_wg, "WGAN-GP — Training Progression", "wgan_progression.png")

## 8.4 Diversity Quantification
We compute three simple metrics to numerically compare mode coverage:
1. **Pixel standard deviation** across a batch — higher = more diverse
2. **Average pairwise LPIPS-like L1 distance** — higher = less mode collapse
3. **Unique colour histogram bins** — richer palette = more diverse

In [ ]:
# ============================================================
# 8.4  Diversity metrics
# ============================================================

def diversity_metrics(generator, z_dim, n_samples=256, device='cuda'):
    """Compute simple pixel-space diversity metrics."""
    generator.eval()
    noise = torch.randn(n_samples, z_dim, 1, 1, device=device)
    with torch.no_grad():
        imgs = generator(noise).cpu()  # (N, 3, 64, 64) in [-1, 1]

    # 1. Per-pixel std, averaged over all pixels
    pixel_std = imgs.std(dim=0).mean().item()

    # 2. Average pairwise L1 distance (sample 500 pairs)
    n_pairs = min(500, n_samples * (n_samples - 1) // 2)
    flat = imgs.view(n_samples, -1)
    idx_a = torch.randint(0, n_samples, (n_pairs,))
    idx_b = torch.randint(0, n_samples, (n_pairs,))
    pairwise_l1 = (flat[idx_a] - flat[idx_b]).abs().mean().item()

    # 3. Colour histogram richness (quantise to 32 bins per channel)
    imgs_uint8 = ((imgs + 1) * 127.5).clamp(0, 255).byte()
    quantised = (imgs_uint8 // 8).view(n_samples, 3, -1)  # 32 bins
    unique_combos = set()
    for img_q in quantised:
        for px in range(0, img_q.size(1), 16):  # sample every 16th pixel
            combo = tuple(img_q[:, px].numpy())
            unique_combos.add(combo)
    
    generator.train()
    return {
        "pixel_std": pixel_std,
        "pairwise_l1": pairwise_l1,
        "colour_combos": len(unique_combos),
    }

m_dc = diversity_metrics(netG_dc, Z_DIM, device=device)
m_wg = diversity_metrics(netG_wg, Z_DIM, device=device)

print(f"{'Metric':<25} {'DCGAN':>10} {'WGAN-GP':>10}")
print("-" * 47)
for k in m_dc:
    print(f"{k:<25} {m_dc[k]:>10.4f} {m_wg[k]:>10.4f}")

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, k in zip(axes, m_dc):
    vals = [m_dc[k], m_wg[k]]
    bars = ax.bar(["DCGAN", "WGAN-GP"], vals, color=["#4A90D9", "#50C878"], edgecolor="black")
    ax.set_title(k.replace("_", " ").title(), fontsize=12)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{v:.3f}", ha='center', va='bottom', fontsize=10)
plt.suptitle("Diversity Comparison — Higher is Better", fontsize=14, y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "diversity_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

# 9. Interactive App Deployment (Gradio)
A Gradio application allowing:
- Choosing between DCGAN and WGAN-GP (or side-by-side comparison)
- Controlling the random seed
- Adjusting the number of generated samples
- Viewing real training images for reference

In [ ]:
# ============================================================
# 9. Gradio App
# ============================================================
import gradio as gr

def generate(model_name, seed, num_images):
    """Generate images from a trained generator."""
    num_images = int(num_images)
    torch.manual_seed(int(seed))
    noise = torch.randn(num_images, Z_DIM, 1, 1, device=device)

    with torch.no_grad():
        if model_name == "DCGAN":
            netG_dc.eval()
            imgs = netG_dc(noise).cpu()
            netG_dc.train()
        else:
            netG_wg.eval()
            imgs = netG_wg(noise).cpu()
            netG_wg.train()

    grid = vutils.make_grid(imgs, nrow=int(num_images**0.5), padding=2, normalize=True)
    arr = grid.mul(255).add_(0.5).clamp_(0, 255).permute(1, 2, 0).numpy().astype(np.uint8)
    return Image.fromarray(arr)


def compare(seed, num_images):
    """Generate from both models with the same seed for direct comparison."""
    img_dc = generate("DCGAN", seed, num_images)
    img_wg = generate("WGAN-GP", seed, num_images)
    return img_dc, img_wg


with gr.Blocks(title="GAN Mode Collapse Demo", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## 🎨 GAN Image Generator — DCGAN vs WGAN-GP\n"
        "Explore how Wasserstein loss + gradient penalty eliminates mode collapse "
        "and produces more diverse, higher-quality samples."
    )

    with gr.Tab("Single Model"):
        with gr.Row():
            with gr.Column(scale=1):
                model_dd = gr.Dropdown(["DCGAN", "WGAN-GP"], value="WGAN-GP", label="Model")
                seed_sl  = gr.Slider(0, 99999, value=42, step=1, label="Random Seed")
                num_sl   = gr.Slider(4, 64, value=16, step=4, label="Number of Images")
                gen_btn  = gr.Button("🚀 Generate", variant="primary")
            with gr.Column(scale=2):
                out_img  = gr.Image(label="Generated Samples", type="pil")
        gen_btn.click(fn=generate, inputs=[model_dd, seed_sl, num_sl], outputs=out_img)

    with gr.Tab("Side-by-Side Comparison"):
        gr.Markdown("Generate from **both** models using the **same** random seed to directly compare diversity.")
        with gr.Row():
            with gr.Column(scale=1):
                seed_sl2 = gr.Slider(0, 99999, value=42, step=1, label="Random Seed")
                num_sl2  = gr.Slider(4, 64, value=16, step=4, label="Number of Images")
                cmp_btn  = gr.Button("⚡ Compare", variant="primary")
            with gr.Column(scale=1):
                out_dc   = gr.Image(label="DCGAN Output", type="pil")
            with gr.Column(scale=1):
                out_wg   = gr.Image(label="WGAN-GP Output", type="pil")
        cmp_btn.click(fn=compare, inputs=[seed_sl2, num_sl2], outputs=[out_dc, out_wg])

try:
    demo.launch(share=True, inline=False)
except Exception as e:
    print(f"Gradio launch failed: {e}")
    print("Try running: demo.launch(inline=True)")

# 10. Summary & Conclusions

## Key Findings

| Aspect | DCGAN | WGAN-GP |
|--------|-------|---------|
| **Training stability** | Oscillating D/G losses; D can overpower G | Smooth, monotonic Wasserstein distance increase |
| **Mode collapse** | Often generates repetitive, similar images | Produces diverse outputs across the data manifold |
| **Loss interpretability** | BCE loss doesn't correlate with image quality | Wasserstein distance directly measures distribution gap |
| **Convergence** | No reliable convergence signal | W distance converges = distributions converging |

## Why WGAN-GP Works
1. **Wasserstein loss** provides meaningful gradients everywhere (no vanishing gradients when D is too good)
2. **Gradient penalty** softly enforces the 1-Lipschitz constraint, keeping the critic well-behaved
3. **Multiple critic updates** per generator step ensure the critic provides an accurate distance estimate
4. **No Sigmoid** output means the critic can express unbounded scores, avoiding saturation

## Files Saved
- `dcgan_G_final.pth` / `dcgan_D_final.pth` — DCGAN checkpoints
- `wgan_G_final.pth` / `wgan_C_final.pth` — WGAN-GP checkpoints
- `loss_curves.png` — Training loss comparison
- `comparison_final.png` — Side-by-side generated samples
- `diversity_comparison.png` — Quantitative diversity metrics